# 📈 Linear Regression Workshop
Welcome to the Linear Regression workshop! We'll start from scratch and move to Scikit-Learn.

## 1. Problem Overview
We want to predict a continuous target variable (house prices, stock prices, disease progression) based on one or more input features.

## 2. Dataset Exploration
Let's load the Diabetes dataset from Scikit-Learn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes

data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
display(df.head())

## 3. Data Preprocessing
Scaling is important for Gradient Descent.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[['bmi']].values # Using single feature for visualization first
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. From Scratch Implementation
A pure Python implementation using iterative gradient descent.

In [ ]:
class PurePythonLR:
    def __init__(self, lr=0.1, epochs=100):
        self.lr = lr
        self.epochs = epochs
        self.w = 0.0
        self.b = 0.0
        
    def fit(self, X, y):
        n = len(X)
        for _ in range(self.epochs):
            y_pred = [self.w * x[0] + self.b for x in X]
            dw = (-2/n) * sum([x[0] * (yi - ypi) for x, yi, ypi in zip(X, y, y_pred)])
            db = (-2/n) * sum([(yi - ypi) for yi, ypi in zip(y, y_pred)])
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict(self, X):
        return [self.w * x[0] + self.b for x in X]

model_scratch = PurePythonLR()
model_scratch.fit(X_train_scaled, y_train)
print(f'Weight: {model_scratch.w}, Bias: {model_scratch.b}')

## 5. NumPy Implementation
Vectorized for all features.

In [ ]:
class NumpyLR:
    def __init__(self, lr=0.1, epochs=1000):
        self.lr = lr
        self.epochs = epochs
        
    def fit(self, X, y):
        X_b = np.c_[np.ones((len(X), 1)), X]
        self.theta = np.zeros(X_b.shape[1])
        m = len(X)
        
        for _ in range(self.epochs):
            gradients = 2/m * X_b.T.dot(X_b.dot(self.theta) - y)
            self.theta -= self.lr * gradients
            
    def predict(self, X):
        X_b = np.c_[np.ones((len(X), 1)), X]
        return X_b.dot(self.theta)

np_model = NumpyLR()
np_model.fit(X_train_scaled, y_train)

## 6. Scikit-Learn Workflow

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

sk_model = LinearRegression()
sk_model.fit(X_train_scaled, y_train)
preds = sk_model.predict(X_test_scaled)
print('MSE:', mean_squared_error(y_test, preds))
print('R2:', r2_score(y_test, preds))

## 7. Visualization Lab

In [ ]:
plt.scatter(X_test_scaled, y_test, color='black', label='Data')
plt.plot(X_test_scaled, preds, color='blue', linewidth=3, label='Prediction')
plt.xlabel('BMI (Scaled)')
plt.ylabel('Disease Progression')
plt.legend()
plt.show()

## 8. Hyperparameter Experiments
Compare learning rates.

In [ ]:
lrs = [0.001, 0.01, 0.1, 0.5]
print('Testing different learning rates...')
for lr in lrs:
    m = NumpyLR(lr=lr, epochs=50)
    m.fit(X_train_scaled, y_train)
    print(f'LR: {lr}, final parameters sum: {np.sum(m.theta)}')

## 9. Failure Cases
Linear regression fails on non-linear data.

In [ ]:
X_fail = np.linspace(-10, 10, 100).reshape(-1, 1)
y_fail = X_fail**2 + np.random.randn(100, 1)*10

bad_model = LinearRegression().fit(X_fail, y_fail)
plt.scatter(X_fail, y_fail)
plt.plot(X_fail, bad_model.predict(X_fail), color='red')
plt.title('Linear Model Failing on Quadratic Data')
plt.show()